In [7]:
# value objects

class Value:
    # ==============================init==============================
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
    # ==============================/init==============================

    # ==============================make_the_output_nice==============================
    def __repr__(self):
        return f"Value(data={self.data})"
    # ==============================/make_the_output_nice==============================

    # ==============================add==============================
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out= Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other  
    # ==============================/add==============================

    # ==============================sub==============================
    def __neg__(self):
            return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    # ==============================/sub==============================
    
    # ==============================mul==============================
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out= Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out 
    
    def __rmul__(self, other):
        return self * other  
    # ==============================/mul==============================
    
    # ==============================power==============================
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only support int/float powers for now"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward =_backward

        return out
    # ==============================/power==============================

    # ==============================div==============================
    def __truediv__(self, other):
        return self * other**-1
    # ==============================/div==============================    

    # ==============================tanh==============================
    def tanh(self):
        x= self.data
        t= ((math.exp(2*x)-1)/ (math.exp(2*x)+1))
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1-t**2) * out.grad
        out._backward = _backward

        return out
    # ==============================/tanh==============================
    # ==============================exp==============================
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    # ==============================/exp==============================
    # ==============================backward_propagation==============================
    def backward(self):
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        topo
    # ==============================/backward_propagation==============================
    

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
     

In [22]:
# imports

import math
import numpy as np
import matplotlib.pyplot as plt
import random
from graphviz import Digraph

In [23]:
def trace(root):
    #builds a set of all nodes and edges ina graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR=left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph create a rectangular('record') node for it
        dot.node(name= uid, label="{%s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape = 'record')
        if n._op:
            # if this value is the result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # and connect this node to
            dot.edge(uid+n._op, uid)
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot


In [35]:
import math
from typing import Any
import numpy as np
import matplotlib.pyplot as plt

class Neuron:
    def __init__(self, nin):        # nin = number of inputs (how many inputs per neuron)
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self,x):
        # w * x + b
        # w * x is a dot product
        # call can be used to make python to make a fnction and use function notation such as n(x)
        # for example
        # x = [2.0, 3.0]
        # n = Neuron (2), 2 input neuron
        # n(x) = whatever the function does to it
        
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)   # zip(self.w, x) is like writing them as a set of coordinates, (w,x)
                                                                # start the sum with self.b and not 0.0
        out = act.tanh()
        return out
    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):

        return [p for neuron in self.neurons for p in neuron.parameters()] 
        
        # WHAT THE CODE ACTUALLY DOES:

        # params = []
        # for neuron in self.neurons:
        #     ps = neuron.parameters()
        #     params.extend(ps)
        # return params

# multilayer perceptron
class MLP: 
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]



In [113]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1] )
n(x)

Value(data=0.17460844957773)

In [125]:
xs  = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] # desired targets

# basically we are telling it to give 1.0 when it is fed the first row, -1 when fed the second row and so on and so forth



In [131]:
for k in range(5000):
    # forward pass
    ypred = [n(x) for x in xs]
    # evaluate loss
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

    # backward pass
    for p in n.parameters():
        p.grad = 0.0
        
    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)

0 0.04753404178167296
1 0.04249163549885157
2 0.0383900292654138
3 0.0349906639897418
4 0.032129052852248347
5 0.0296881485133465
6 0.027582424465668387
7 0.025747957556610346
8 0.024136028971169437
9 0.02270886943006315
10 0.02143675529350849
11 0.020295980752109317
12 0.019267412810962326
13 0.018335442801504512
14 0.017487213175562278
15 0.016712038900040953
16 0.016000968687268094
17 0.015346448213599997
18 0.014742058739249029
19 0.01418231217124017
20 0.013662488864602402
21 0.013178508128502233
22 0.01272682400571786
23 0.01230434076099902
24 0.011908343869746147
25 0.01153644329393621
26 0.011186526570638936
27 0.010856719791464494
28 0.010545354969161966
29 0.010250942606028768
30 0.009972148523397912
31 0.009707774200768396
32 0.009456740020680742
33 0.00921807093120008
34 0.008990884129271771
35 0.008774378440824598
36 0.008567825131501167
37 0.008370559928492635
38 0.008181976071575878
39 0.008001518241979776
40 0.007828677242594995
41 0.0076629853234240755
42 0.00750401206

In [ ]:
ypred

[Value(data=0.997601028485684),
 Value(data=-0.9968436772052401),
 Value(data=-0.9942478279607686),
 Value(data=0.9958997271900717)]

BELOW IS THE MANUAL STEPS COMMENTED. ABOVE IS THE ITERATIVE PROCESS

In [ ]:
# ypred = [n(x) for x in xs]

# loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])
# loss

Value(data=1.1610229786386193e-05)

In [ ]:
# loss.backward()

In [ ]:
# for p in n.parameters():
#     p.data += -0.01 * p.grad
#     # we do a negative because the gradient points in the direction of the vector that increases the gradient
#     # so we need to move in the opposite direction to minimise the loss
    

In [ ]:
# ypred

[Value(data=0.997368132648258),
 Value(data=-0.9997026623997697),
 Value(data=-0.9996711580490737),
 Value(data=0.9978817560688106)]

In [ ]:
# n.parameters()

[Value(data=0.4213758711986859),
 Value(data=-0.24280737737113986),
 Value(data=1.4427246028227565),
 Value(data=0.12011059455143627),
 Value(data=0.7735923869725687),
 Value(data=1.0557421065948756),
 Value(data=-0.6243658532277808),
 Value(data=1.0192614159312754),
 Value(data=-0.15608500471512174),
 Value(data=0.42415341469200896),
 Value(data=0.07466115755342068),
 Value(data=-0.5235398874357303),
 Value(data=-0.3477116748076221),
 Value(data=1.3298169098830561),
 Value(data=-1.288096596350886),
 Value(data=0.09867763614546526),
 Value(data=-1.201115647901367),
 Value(data=-1.262713149571941),
 Value(data=-2.409193191212864),
 Value(data=-2.406088852191423),
 Value(data=-1.8171137703593048),
 Value(data=-0.5106010687478122),
 Value(data=-0.04118681026666968),
 Value(data=-0.413218266643571),
 Value(data=0.9134386292688317),
 Value(data=0.11391667084749252),
 Value(data=-1.8037399476456308),
 Value(data=-0.28005980475403236),
 Value(data=0.5247724757688341),
 Value(data=2.2293632523